In [2]:
def get_building_energy(energy_csv, training_csv):
    import pandas as pd
    import numpy as np
    import geopandas as gpd
    from shapely.geometry import Point
    
    energy = pd.read_csv(energy_csv)

    selected_columns = [
    "Latitude",
    "Longitude",
    "Weather Normalized Site EUI (kBtu/ft²)",
    "Weather Normalized Site Energy Use (kBtu)",
    "Electricity Use - Grid Purchase (kWh)",
    "Natural Gas Use (kBtu)",
    "ENERGY STAR Score"
    ]

    available_columns = [col for col in selected_columns if col in energy.columns]
    df_selected = energy[available_columns]
    df_selected.replace('Not Available', np.nan, inplace=True)
    df_selected.dropna(inplace=True)
    df_selected = df_selected.apply(pd.to_numeric, errors='coerce')

    df1 = df_selected
    df2 = pd.read_csv(training_csv)

    # Example dataframes (replace with your actual data)
    # df1: Contains energy use metrics and coordinates
    # df2: Contains only coordinates
    # Assume columns 'longitude' and 'latitude' in both dataframes

    # Create GeoDataFrames from df1 and df2
    gdf1 = gpd.GeoDataFrame(
        df1,
        geometry=gpd.points_from_xy(df1.Longitude, df1.Latitude),
        crs="EPSG:4326"  # Assuming original coordinates are in WGS84
    )

    gdf2 = gpd.GeoDataFrame(
        df2,
        geometry=gpd.points_from_xy(df2.Longitude, df2.Latitude),
        crs="EPSG:4326"
    )

    # Project to a CRS with meters as units (e.g., EPSG:3857 or a local UTM)
    gdf1 = gdf1.to_crs(epsg=3857)
    gdf2 = gdf2.to_crs(epsg=3857)

    # Create a buffer of 50 meters around each point in gdf2
    gdf2['buffer'] = gdf2.geometry.buffer(200)

    # It is often useful to work with the buffer as the geometry for spatial join
    gdf2_buffer = gdf2.copy()
    gdf2_buffer.set_geometry('buffer', inplace=True)

    # Perform a spatial join: find all gdf1 points within each buffer from gdf2
    # The 'op' parameter has been renamed to 'predicate' in recent versions of GeoPandas.
    joined = gpd.sjoin(gdf1, gdf2_buffer, predicate='within')

    # Now, joined contains df1 rows with an index reference (index_right) to gdf2_buffer.
    # Group by the df2 index (index_right) and compute the mean for each energy metric column.
    # Replace 'Natural Gas Use (kBtu)' with your actual energy use metric columns as needed.
    energy_columns = ['Weather Normalized Site EUI (kBtu/ft²)',
        'Weather Normalized Site Energy Use (kBtu)',
        'Electricity Use - Grid Purchase (kWh)', 'Natural Gas Use (kBtu)',
        'ENERGY STAR Score']  # add other metric columns here

    aggregated = joined.groupby('index_right')[energy_columns].mean().reset_index()

    result = gdf2.merge(aggregated, left_index=True, right_on='index_right', how='left')

    result = result.drop(columns=['buffer'])

    result = result.to_crs(epsg=4326)
    result = result.drop(columns=['geometry'])

    result.set_index("index_right", inplace=True)
    result.fillna(0, inplace=True)
    return(result)

In [6]:
energy = 'Energy_and_Water_Data_Disclosure_for_Local_Law_84_2022__Data_for_Calendar_Year_2021__20250304.csv'
training = 'Training_data_uhi_index_UHI2025-v2.csv'
validation = 'Submission_template_UHI2025-v2.csv'

energy_df = get_building_energy(energy, validation)
energy_df

/var/folders/mc/xgzr57gj1wqb_286q_dzw53c0000gn/T/ipykernel_97029/3102802907.py:7: DtypeWarning: Columns (9,15,216,217) have mixed types. Specify dtype option on import or set low_memory=False.
  energy = pd.read_csv(energy_csv)
/var/folders/mc/xgzr57gj1wqb_286q_dzw53c0000gn/T/ipykernel_97029/3102802907.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected.replace('Not Available', np.nan, inplace=True)
/var/folders/mc/xgzr57gj1wqb_286q_dzw53c0000gn/T/ipykernel_97029/3102802907.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected.dropna(inplace=True)


,Longitude,Latitude,UHI Index,Weather Normalized Site EUI (kBtu/ft²),Weather Normalized Site Energy Use (kBtu),Electricity Use - Grid Purchase (kWh),Natural Gas Use (kBtu),ENERGY STAR Score
index_right,,,,,,,,
0,-73.971665,40.788763,0.0,81.928571,9.305911e+06,1.082044e+06,4.350930e+06,55.000000
1,-73.971928,40.788875,0.0,80.980000,9.741772e+06,1.084841e+06,4.804590e+06,56.600000
2,-73.967080,40.789080,0.0,74.572222,1.027143e+07,5.254804e+05,6.545425e+06,76.111111
3,-73.972550,40.789082,0.0,81.261538,8.712015e+06,9.241308e+05,4.533307e+06,59.153846
4,-73.969697,40.787953,0.0,74.700000,9.875366e+06,1.177865e+06,3.851059e+06,58.222222
...,...,...,...,...,...,...,...,...
1035,-73.919388,40.813803,0.0,72.900000,7.017191e+06,6.785009e+05,2.179153e+06,65.800000
1036,-73.931033,40.833178,0.0,78.420000,4.154236e+06,2.498516e+05,2.628852e+06,71.600000
1037,-73.934647,40.854542,0.0,82.985714,5.788453e+06,3.388577e+05,3.354008e+06,66.642857
